# Gold Serving Table: Station Arrival Summary

Build station and line arrival metrics for the latest available London operating date.

**Source:** `workspace.urbanpulse_gold.fact_arrival_observation`

**Dimensions:**
- `workspace.urbanpulse_gold.dim_station`
- `workspace.urbanpulse_gold.dim_line`

**Target:** `workspace.urbanpulse_gold.station_arrival_summary`

**Grain:** One station and line summary for the latest available arrival observation date.

## 1. Project paths

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print(f"Project root: {PROJECT_ROOT}")

## 2. Imports

In [0]:
from pyspark.sql import functions as F

from urbanpulse.transformations.station_arrival_summary import (
    build_station_arrival_summary,
)

from urbanpulse.quality.station_arrival_summary import (
    invalid_station_arrival_summary,
)

## 3. Table Names

In [0]:
FACT_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "fact_arrival_observation"
)

DIM_STATION_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "dim_station"
)

DIM_LINE_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "dim_line"
)

TARGET_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "station_arrival_summary"
)

## 4. Read source gold tables

In [0]:
fact_df = spark.table(
    FACT_TABLE
)

dim_station_df = spark.table(
    DIM_STATION_TABLE
)

dim_line_df = spark.table(
    DIM_LINE_TABLE
)

print(
    f"Arrival fact rows: "
    f"{fact_df.count()}"
)

## 5. Build serving dataset

In [0]:
summary_df = (
    build_station_arrival_summary(
        fact_df=fact_df,
        dim_station_df=dim_station_df,
        dim_line_df=dim_line_df,
    )
)

summary_count = (
    summary_df.count()
)

print(
    f"Station-line summaries: "
    f"{summary_count}"
)

display(
    summary_df
    .orderBy(
        "station_name",
        "line_name",
    )
)

## 6. Validate serving grain

In [0]:
duplicate_grain_df = (
    summary_df
    .groupBy(
        "station_key",
        "line_key",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

if duplicate_grain_df.count() > 0:
    display(
        duplicate_grain_df
    )

    raise ValueError(
        "Duplicate station-line summaries detected."
    )

print(
    "Serving grain validation passed."
)

## 7. Apply Quality checks

In [0]:
invalid_df = (
    invalid_station_arrival_summary(
        summary_df
    )
)

invalid_count = (
    invalid_df.count()
)

print(
    f"Invalid rows: "
    f"{invalid_count}"
)

if invalid_count > 0:
    display(
        invalid_df
    )

    raise ValueError(
        f"{invalid_count} invalid station "
        "arrival summary rows detected."
    )

print(
    "Station arrival summary quality passed."
)

## 8. Validate ETA consistency

In [0]:
invalid_eta_df = (
    summary_df
    .filter(
        (
            F.col("min_eta_seconds")
            >
            F.col("avg_eta_seconds")
        )
        |
        (
            F.col("avg_eta_seconds")
            >
            F.col("max_eta_seconds")
        )
    )
)

if invalid_eta_df.count() > 0:
    display(
        invalid_eta_df
    )

    raise ValueError(
        "Invalid ETA aggregation detected."
    )

print(
    "ETA aggregation validation passed."
)

## 9. Add serving metadata

In [0]:
serving_df = (
    summary_df
    .withColumn(
        "serving_updated_at",
        F.current_timestamp(),
    )
)

## 10. Write Serving Table

In [0]:
(
    serving_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        TARGET_TABLE
    )
)

print(
    f"Created serving table: "
    f"{TARGET_TABLE}"
)

## 11. Verify table

In [0]:
%sql
SELECT
    station_name,
    line_name,
    arrival_observations,
    distinct_vehicles,
    avg_eta_seconds,
    min_eta_seconds,
    max_eta_seconds,
    next_expected_arrival_local,
    latest_prediction_timestamp_local
FROM workspace.urbanpulse_gold.station_arrival_summary
ORDER BY
    station_name,
    line_name;

## 12. Check uniqueness

In [0]:
%sql
SELECT
    station_key,
    line_key,
    COUNT(*) AS records
FROM workspace.urbanpulse_gold.station_arrival_summary
GROUP BY
    station_key,
    line_key
HAVING COUNT(*) > 1;

## 13. Check Dimension reference

In [0]:
%sql
SELECT s.*
FROM workspace.urbanpulse_gold.station_arrival_summary s

LEFT ANTI JOIN workspace.urbanpulse_gold.dim_station d
    ON s.station_key = d.station_key;

In [0]:
%sql
SELECT s.*
FROM workspace.urbanpulse_gold.station_arrival_summary s

LEFT ANTI JOIN workspace.urbanpulse_gold.dim_line d
    ON s.line_key = d.line_key;

## 14. Check current dimension only

In [0]:
%sql
SELECT
    s.station_name,
    s.line_name
FROM workspace.urbanpulse_gold.station_arrival_summary s

INNER JOIN workspace.urbanpulse_gold.dim_station ds
    ON s.station_key = ds.station_key

INNER JOIN workspace.urbanpulse_gold.dim_line dl
    ON s.line_key = dl.line_key

WHERE
    ds.is_current = FALSE
    OR dl.is_current = FALSE;

## 15. Dashboard KPI query

In [0]:
%sql
SELECT
    station_name,

    SUM(
        arrival_observations
    ) AS arrival_observations,

    SUM(
        distinct_vehicles
    ) AS observed_vehicles,

    ROUND(
        AVG(avg_eta_seconds),
        1
    ) AS avg_eta_seconds

FROM workspace.urbanpulse_gold.station_arrival_summary

GROUP BY
    station_key,
    station_name

ORDER BY
    arrival_observations DESC;

In [0]:
%sql
-- Next arrival view
SELECT
    station_name,
    line_name,
    next_expected_arrival_local,
    min_eta_seconds,
    latest_prediction_timestamp_local
FROM workspace.urbanpulse_gold.station_arrival_summary
ORDER BY
    next_expected_arrival_local;

In [0]:
%sql
-- Check data freshness
SELECT
    MIN(
        latest_prediction_timestamp_utc
    ) AS oldest_prediction,

    MAX(
        latest_prediction_timestamp_utc
    ) AS newest_prediction,

    MAX(
        serving_updated_at
    ) AS serving_updated_at

FROM workspace.urbanpulse_gold.station_arrival_summary;

In [0]:
%sql
SELECT COUNT(*) AS rows
FROM workspace.urbanpulse_gold.station_arrival_summary;